# Condition datasets — verification

Per condition, answer two questions:

1. **Is the intended impairment present at the intended magnitude?** Every component of
   θ is re-estimated *from the data* and compared against the θ the file declares in its
   metadata — so the check is independent of the code that generated it.
2. **Did anything else change?** Shape, dtype, finiteness, labels, frame order, and — for
   the baseline — bit-for-bit equality with the clean subset.

Ordering note: nothing here is normalized. `make_subset.py` copies `X` raw and `src/data.py`
applies `unit_power` at **load** time, so injection happens strictly before normalization,
which is the physical order (hardware chain first).

In [ ]:
import sys, pathlib
# Find the repo root (the folder containing `src/`) and put it FIRST on the import path,
# so this works whether the notebook is launched from notebooks/ or the repo root.
ROOT = pathlib.Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# Drop any stale/namespace `scripts`/`src` cached by an earlier failed import.
for _m in [m for m in sys.modules if m == 'scripts' or m.startswith('scripts.')
           or m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]

assert (ROOT / 'scripts' / 'check_condition.py').exists(), f"not the repo root: {ROOT}"
print('repo root:', ROOT)

import json
import numpy as np
import matplotlib.pyplot as plt

CONFIG = 'baseline_100'   # its data.path is the clean subset
SNR = 30                  # estimator variance falls with channel noise, so check high SNR

## The conditions and their θ

Defined in `configs/conditions.yaml`. Operator lists read **along the signal chain**
(phase noise first, ADC last); the thesis notation `Q_b ∘ B_d ∘ G_{a,ψ} ∘ P_σ` reads the
other way. Same chain, opposite writing direction.

In [ ]:
from scripts.make_condition import load_conditions, load_sample_rate_hz
from src.distortions import build_compose

# f_s is declared once for the whole file: it converts the datasheet phase-noise figure into
# the per-sample sigma_w below, and is recorded in every generated dataset's attrs.
table = load_conditions()
sample_rate_hz = load_sample_rate_hz()
print(f"sample_rate_hz = {sample_rate_hz:,.0f}\n")
for name, specs in table.items():
    chain = build_compose(specs, sample_rate_hz=sample_rate_hz)
    print(f"{name:22s} identity={str(chain.is_identity):5s}  {json.dumps(chain.params())}")

## Generate

One frozen file per condition, generated once and reused across every training seed.
Each is roughly the size of the clean subset (~2 GB), so this is off by default.
Equivalent CLI: `python scripts/make_condition.py --condition phase_noise --config baseline_100`

In [ ]:
from scripts.make_condition import make_condition

GENERATE = False              # flip to True to (re)generate
CONDITIONS = ['baseline', 'phase_noise', 'iq_imbalance', 'quantization', 'all']

if GENERATE:
    for condition in CONDITIONS:
        make_condition(condition, config=CONFIG, overwrite=False)
else:
    print('skipped; set GENERATE = True to write the datasets')

In [ ]:
from scripts.check_condition import check_condition

report = check_condition('all', config=CONFIG, snr=SNR, frame=0)
plt.show()

In [ ]:
for condition in ['iq_imbalance', 'quantization', 'all']:
    check_condition(condition, config=CONFIG, snr=SNR, frame=0)
    plt.show()
    print()

## Cross-condition summary

`APPROX` marks the composite chain: each estimator inverts a single operator, so with
several active at once they bias each other. Those rows are informative, not assertions —
the single-impairment conditions are where recovery is exact.

In [ ]:
from scripts.make_condition import default_output_name
from src.config import resolve_data_path

# `baseline` is normally absent: identity theta makes it a 2 GB byte-for-byte copy of the
# source, and that round-trip is asserted in tests/test_make_condition.py instead.
clean_path = pathlib.Path(resolve_data_path(CONFIG)[0])

rows = []
for condition in CONDITIONS:
    if not (clean_path.parent / default_output_name(clean_path, condition, False)).exists():
        rows.append((condition, 'file not generated', '', '', 'SKIP'))
        continue
    r = check_condition(condition, config=CONFIG, snr=SNR, plot=False, verbose=False)
    for guard, ok in r['guards']:
        rows.append((condition, guard, '', '', 'PASS' if ok else 'FAIL'))
    for p in r['recovered']:
        rows.append((condition, p['parameter'], f"{p['declared']:.4g}",
                     f"{p['recovered']:.4g}", p['status']))

print(f"{'condition':<14}{'check':<32}{'declared':>10}{'recovered':>12}   status")
for condition, name, declared, recovered, status in rows:
    print(f"{condition:<14}{name:<32}{declared:>10}{recovered:>12}   {status}")

## How much energy each condition injects

`10*log10(mean|y - x|^2 / mean|x|^2)` per frame, on the same rows of both files, median over
one frame from **every (class, SNR) cell** — a leading slice of rows would be a single cell,
so its median would report that cell's PAPR rather than the dataset's.

No models are involved, and nothing here calls the operators. The `expected` column is
re-derived from the θ each impaired file *declares*, by hand from the physics, so a gap wider
than `TOL_DB` means the file does not match its own metadata.

In [ ]:
import h5py
from scripts.make_condition import default_output_name
from src.config import resolve_data_path
from src.data import KEY_X, read_labels_and_snr

POWER_CONDITIONS = ['phase_noise', 'phase_noise_exaggerated', 'iq_imbalance',
                    'quantization', 'all']
TOL_DB = 3.0   # wider than any estimator spread here, so a flag is a generator bug


def perturbation_db(clean, dirty):
    """Per-frame injected power relative to frame power, in dB. Sums, not means: the
    denominators cancel, so this is exactly mean|y-x|^2 / mean|x|^2."""
    c, d = clean.astype(np.float64), dirty.astype(np.float64)
    return 10.0 * np.log10(((d - c) ** 2).sum((1, 2)) / (c ** 2).sum((1, 2)))


def quantization_ratio(frames, n_bits, percentile):
    """Median ADC error power per frame power, from b and the reference percentile alone.
    Delta^2/12 for the bulk plus the clipping the percentile reference deliberately allows --
    at 99.9 only ~2 samples per frame clip, but each misses by far more than Delta/2."""
    ratios = []
    for c in frames.astype(np.float64):
        branches = np.abs(np.concatenate((c[:, 0], c[:, 1])))
        fs = np.percentile(branches, percentile)
        delta = 2.0 * fs / 2 ** n_bits
        # A clipped sample lands on the outermost code centre, FS - Delta/2, not within +/-Delta/2.
        err2 = np.where(branches > fs, (branches - (fs - delta / 2.0)) ** 2, delta ** 2 / 12.0)
        ratios.append(err2.sum() / (branches ** 2).sum())
    return float(np.median(ratios))


def expected_parts(theta, frames, frame_len):
    """Analytic injected-power ratio per operator of theta. The terms are independent, so
    they add in linear power; each is derived here from the physics, not from src.distortions."""
    parts = {}
    if 'PhaseNoise' in theta:
        # |x e^{j phi} - x|^2 = |x|^2 * 2(1 - cos phi), and phi_n is a Wiener walk pinned at
        # phi_0 = 0, so var(phi_n) = n sigma_w^2 and E[cos phi_n] = exp(-n sigma_w^2 / 2).
        # Small-angle this is just sigma_w^2 (T-1)/2 -- the mean square accumulated phase.
        sigma_w = theta['PhaseNoise']['sigma_w']
        n = np.arange(frame_len)
        parts['phase'] = float(np.mean(2.0 * (1.0 - np.exp(-n * sigma_w ** 2 / 2.0))))
    if 'IQImbalance' in theta:
        # y = alpha x + beta conj(x), so the error is (alpha-1) x + beta conj(x). The two have
        # EXACTLY equal magnitude, which is why this sits 3 dB above the image-only -IRR figure.
        g = 10.0 ** (theta['IQImbalance']['gain_db'] / 20.0)
        psi = np.deg2rad(theta['IQImbalance']['phase_deg'])
        alpha, beta = (1 + g * np.exp(1j * psi)) / 2, (1 - g * np.exp(-1j * psi)) / 2
        parts['image'] = float(abs(alpha - 1) ** 2 + abs(beta) ** 2)
    if theta.get('Quantize', {}).get('n_bits') is not None:
        reference = theta['_reference']
        assert reference['kind'] == 'percentile', f"unhandled reference {reference['kind']!r}"
        # FS is read off the CLEAN frame; under `all` the declared measurement point is after
        # G_{a,psi}, which moves it by a fraction of a dB and this term by no more than that.
        parts['adc'] = quantization_ratio(frames, theta['Quantize']['n_bits'],
                                          reference['percentile'])
    return parts


clean_path = pathlib.Path(resolve_data_path(CONFIG)[0])
class_idx, snr_all, frame_len, _ = read_labels_and_snr(clean_path)

# One frame from every (class, SNR) cell: group by the cell, then take a random member of each.
cell = class_idx.astype(np.int64) * 1000 + snr_all.astype(np.int64)
order = np.argsort(cell, kind='stable')
starts = np.flatnonzero(np.r_[True, np.diff(cell[order]) != 0])
counts = np.diff(np.r_[starts, cell.size])
pick = np.sort(order[starts + np.random.default_rng(0).integers(0, counts)]).tolist()

with h5py.File(clean_path, 'r') as f:
    clean = f[KEY_X][pick]

print(f"{len(pick)} frames -- one per (class, SNR) cell -- from {clean_path.name}\n")
print(f"{'condition':<24}{'measured':>10}{'expected':>10}{'diff':>8}   terms (dB)")
for condition in POWER_CONDITIONS:
    dirty_path = clean_path.parent / default_output_name(clean_path, condition, False)
    if not dirty_path.exists():
        print(f"{condition:<24}{'-- not generated --':>28}")
        continue
    with h5py.File(dirty_path, 'r') as f:
        theta = json.loads(f.attrs.get('theta', '{}'))
        dirty = f[KEY_X][pick]
    measured = float(np.median(perturbation_db(clean, dirty)))
    parts = expected_parts(theta, clean, frame_len)
    expected = 10.0 * np.log10(sum(parts.values()))
    terms = ', '.join(f"{name} {10 * np.log10(value):.1f}" for name, value in parts.items())
    flag = '  <-- FLAG' if abs(measured - expected) > TOL_DB else ''
    print(f"{condition:<24}{measured:>10.2f}{expected:>10.2f}{measured - expected:>+8.2f}"
          f"   {terms}{flag}")

print(f"\n'expected' is analytic, from the theta each FILE declares; 'terms' are its components,")
print(f"summed in power. Flagged past {TOL_DB:.0f} dB, which would mean the generator is not doing")
print("what its parameters claim. Note the image term reads 3 dB above the -IRR figure: IRR")
print("counts only beta*conj(x), and (alpha-1)*x has exactly the same magnitude.")

### Perturbation power and domain shift rank differently

This table measures how much energy `R_θ` injects. It is **not** a prediction of the accuracy
drops in `04_condition_vs_baseline` / `05_all_conditions`, and the two orders disagree — which
is the useful part.

Phase noise is the largest perturbation by power (`phase_noise_exaggerated` at ~-13 dB, an
order of magnitude above the IQ image) and it moves the domain the least. A Wiener walk
multiplies the frame by `e^{jφ_n}`; over one frame that is a slow, near-common rotation, and
`R_0` already draws a uniform initial phase for every frame. The rotation therefore adds no
structure the clean distribution does not already contain — the model has seen every phase.

The image is the opposite. `β·conj(x)` puts energy at the **mirror** frequency, where the
clean signal had none, so it is new in kind rather than merely large: a feature the training
distribution never populates at all. Power says how much was added; shift depends on whether
what was added is something `R_0` could have produced.

(At the datasheet `σ_w = 7.8e-4` the same argument holds but the inversion is invisible — that
condition lands ~9 dB *below* the image in power too, so the ranking only crosses against the
exaggerated variant.)

## Open decision — the AGC reference level

Quantization and DC offset are **not** scale-invariant, so `b` bits and offset `d` only mean
something against a full-scale level. That level is an injectable `ReferenceLevel`
(`peak` / `percentile` / `fixed`) and **the choice is not yet made** — see the TODO in
`configs/conditions.yaml`.

### Where it is measured (settled)

Full scale is a property of the **chain**, not of an operator: `Compose` measures it once per
frame, after `G_{a,ψ}` and before `B_d`, and hands the same number to every consumer. An AGC
reacts to the level at the output of the RF front end; it does not see the DC induced after it,
and it cannot depend on its own quantizer. The practical consequence is that adding `B_d` to a
chain leaves `Q_b`'s step size untouched — without this, the `all` condition would quantize on a
different grid than the `quantization` condition, purely as a side effect of the DC offset.
In the YAML this is the condition-level `reference:` key (with `fs_after:` to move the point).

### Why `fixed` is rejected — and not for the reason it first appears

**Clipping is not the argument.** `FS = 3.5` clips catastrophically (100% of samples in the
loudest 1% of frames), but that shows 3.5 is badly chosen, not that a fixed scale is wrong.
Branch amplitude runs ~1.0 to ~22 across this subset, so setting `FS = 26` clips *nothing*.
The cost simply reappears at the other end: the median frame then exercises ~22 of 256 codes,
so a nominal 8-bit ADC behaves like ~4.5 bits for a typical frame. Recalibrating slides along
that trade-off; it does not remove it.

**The actual reason.** RadioML's ~29 dB inter-frame amplitude spread is an artefact of how the
dataset was *synthesized* — SNR is a separate parameter there — not a property of what arrives
at an antenna. A fixed scale would carry that synthesis artefact into the ADC model disguised
as physics. Per-frame referencing leaves only a dependence on frame *shape*.

**On class dependence.** No class-neutral choice exists, and none is wanted. Under peak
referencing the quantization step scales with the frame peak, so after unit-power
normalization the effective quantization SNR is set by PAPR — which is class-dependent
(3.1–6.8 dB here). That dependence is physically correct and the dataset genuinely preserves
it. The goal is to avoid adding a *spurious* class dependence on top of it, not to remove it.

In [ ]:
import h5py
from src.config import resolve_data_path
from src.data import KEY_X, KEY_Y, MODULATION_CLASSES
from src.distortions import FixedReference, PeakReference, PercentileReference, Quantize, to_complex

N_BITS = 8
clean_path = resolve_data_path(CONFIG)[0]
with h5py.File(clean_path, 'r') as f:
    n = f[KEY_X].shape[0]
    pick = np.sort(np.random.default_rng(0).choice(n, size=1500, replace=False)).tolist()
    frames, classes = f[KEY_X][pick], f[KEY_Y][pick].argmax(1)

peaks = np.abs(frames).max(axis=(1, 2))
print(f"branch peak across frames: min {peaks.min():.2f}  median {np.median(peaks):.2f}  "
      f"max {peaks.max():.2f}\n")

# Both fixed scales are shown on purpose: recalibrating from 3.5 to 26 removes the clipping
# entirely and converts it into starvation, which is the point the table has to make.
strategies = {'peak (per-frame)': PeakReference(),
              'percentile 99.9': PercentileReference(99.9),
              'fixed FS=3.5': FixedReference(3.5),
              'fixed FS=26': FixedReference(26.0)}

# Quantize takes FS as an argument -- here the chain is a single operator, so the measurement
# point is the frame itself and this loop plays the role Compose plays in a real condition.
op = Quantize(N_BITS)
header = f"{'strategy':<19}{'levels: median':>15}{'p1':>6}{'clip% p99':>11}{'loud/quiet':>12}"
print(header)
for label, reference in strategies.items():
    used, clipped, per_class = [], [], {}
    for frame, c in zip(frames, classes):
        x = to_complex(frame)
        fs = reference(x)
        y = op(x, np.random.default_rng(0), fs)
        levels = np.unique(np.concatenate((y.real, y.imag))).size
        used.append(levels)
        clipped.append(np.mean(np.abs(np.concatenate((x.real, x.imag))) > fs) * 100)
        per_class.setdefault(int(c), []).append(levels)
    quiet = min(np.median(v) for v in per_class.values())
    loud = max(np.median(v) for v in per_class.values())
    print(f"{label:<19}{np.median(used):>15.0f}{np.percentile(used, 1):>6.0f}"
          f"{np.percentile(clipped, 99):>11.2f}{loud / max(quiet, 1):>11.1f}x")

print(f"\nideal is {2 ** N_BITS} levels used and 0% clipped.")
print("FS=3.5 clips; FS=26 clips nothing and starves instead -- the trade-off moves, it does")
print("not vanish. The reason to reject fixed is in the markdown above, not in this table.")